In [1]:
%cd ..

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


In [2]:
import os
import glob
import re
from typing import Dict, Any

def get_latest_run_files(base_dir: str = "logs") -> Dict[str, Dict[str, Dict[str, str]]]:
    """
    Scans a directory of method folders, each containing model subfolders with a single
    inner folder, which holds timestamped run files.
    Returns data[method][model] = {'results': path_to_results, 'samples': path_to_samples}
    for the most recent run.
    """
    data: Dict[str, Dict[str, Dict[str, str]]] = {}
    ts_pattern = re.compile(r"(\d{8}_\d{6})_results\.json$")

    for method in os.listdir(base_dir):
        method_path = os.path.join(base_dir, method)
        if not os.path.isdir(method_path):
            continue
        data[method] = {}

        for model in os.listdir(method_path):
            model_path = os.path.join(method_path, model)
            if not os.path.isdir(model_path):
                continue

            # if there's exactly one subdirectory, search inside it:
            children = [
                os.path.join(model_path, d)
                for d in os.listdir(model_path)
                if os.path.isdir(os.path.join(model_path, d))
            ]
            search_dir = children[0] if len(children) == 1 else model_path

            runs: Dict[str, Dict[str, str]] = {}
            for res_file in glob.glob(os.path.join(search_dir, "*_results.json")):
                fn = os.path.basename(res_file)
                m = ts_pattern.match(fn)
                if not m:
                    continue
                ts = m.group(1)

                samp_glob = os.path.join(search_dir, f"{ts}_samples_*.jsonl")
                samp_files = glob.glob(samp_glob)
                if not samp_files:
                    continue

                runs[ts] = {
                    "results": res_file,
                    "samples": samp_files[0]
                }

            if runs:
                latest = max(runs)
                data[method][model] = runs[latest]

        # end for model
    # end for method

    return data


In [3]:
get_latest_run_files()['strokerehab_primitives_1']['internvl_2p5_38b']['results']

'logs/strokerehab_primitives_1/internvl_2p5_38b/OpenGVLab__InternVL2_5-38B/20250702_141225_results.json'

In [4]:
data = get_latest_run_files()
data['strokerehab_primitives_1']['bot']['results']

'logs/strokerehab_primitives_1/bot/20250702_043622_results.json'

In [5]:
from pathlib import Path
import json

def load_results_by_id(log_path):
    log_path = Path(log_path)
    with open(log_path, "r") as f:
        data = json.load(f)

    results = data["results"]
    higher_is_better = data.get("higher_is_better", {})
    configs = data.get("configs", {})

    parsed_metrics = {}

    for task_name, task_results in results.items():
        metric_info = {}
        for k, v in task_results.items():
            if "," in k:
                base_key, _ = k.split(",", 1)
                if base_key.endswith("_stderr"):
                    continue  # we'll pick these up as `*_stderr`
                stderr_key = base_key + "_stderr,combine-and-parse"
                stderr_val = task_results.get(stderr_key, None)

                metric_info[base_key] = {
                    "value": v,
                    "stderr": stderr_val,
                    "higher_is_better": higher_is_better.get(task_name, {}).get(base_key, None)
                }
        parsed_metrics[task_name] = metric_info

    # extract eval-specific flags (optional)
    eval_kwargs = {}
    for task_name, config in configs.items():
        eval_kwargs[task_name] = config.get("lmms_eval_specific_kwargs", {})

    return {
        "metrics": parsed_metrics,
        "eval_kwargs": eval_kwargs,
    }

In [6]:
load_results_by_id(data['strokerehab_primitives_1']['bot']['results'])['metrics']['strokerehab_primitives_1'].keys()

dict_keys(['patient', 'activity', 'id', 'is_in_strokerehab_test_set', 'path_v', 'stroke', 'fps', 'height', 'width', 'duration_s', 'path_l', 'nlabels', 'subsampled_test_set', 'edit_score', 'action_error_rate', 'mae_reach', 'count_reach', 'mae_reposition', 'count_reposition', 'mae_transport', 'count_transport', 'mae_stabilize', 'count_stabilize', 'mae_idle', 'count_idle', 'mae_avg', 'count_truth', 'count_pred'])

In [7]:
data['strokerehab_primitives_1'].keys()

dict_keys(['internvl_2p5_38b', 'bot', 'llava_next_video_7b', 'bot_8f', 'llava_ov_7b', 'internvl_2p5_8b', 'llava_next_video_72b', 'internvl_2p5_2b', 'bot_32f', 'llava_ov_72b', 'llava_ov_0p5b'])

In [8]:
METRICS = {'action_error_rate', 'edit_score', 'mae_avg', 'count_truth', 'count_pred'}

data = get_latest_run_files()
for metric in METRICS:
    print(load_results_by_id(data['strokerehab_primitives_1']['bot']['results'])['metrics']['strokerehab_primitives_1'][metric])


{'value': 0.9230813630813631, 'stderr': 0.025592050404323458, 'higher_is_better': False}
{'value': 7.6918636918636905, 'stderr': 2.5592050404323454, 'higher_is_better': True}
{'value': 1.0, 'stderr': 0.0, 'higher_is_better': None}
{'value': 3.533333333333333, 'stderr': 0.8027729719194863, 'higher_is_better': False}
{'value': 18.666666666666668, 'stderr': 4.013864859597432, 'higher_is_better': None}


In [9]:
from typing import Dict, List, Any

r"""
Build a wide‐format LaTeX table with colored highlighting:

  Model | Prompt | AER ↓ | ES ↑ | MAE ↓ | Pred. Count

- Best values: green background + \textbf{}
- Runner-up: blue background + \textbf{}

Requires LaTeX preamble:
    \usepackage[table]{xcolor}
"""
def generate_model_scenario_table(
    data: Dict[str, Dict[str, Dict[str, str]]],
    scenarios: Dict[str, str],
    models: List[str],
    metric_keys: List[str],
    count_key: str = "count_pred",
    rename: Dict[str, str] = None,
    caption: str = "",
    label: str = ""
) -> str:
    # 1) default renames
    if rename is None:
        rename = {mk: mk.upper() for mk in metric_keys}
    rename[count_key] = "Pred. Count"

    # 2) determine direction (higher_is_better) per metric
    directions: Dict[str, bool] = {}
    for mk in metric_keys:
        directions[mk] = False
        for sc, method in scenarios.items():
            for m in models:
                ent = data.get(method, {}).get(m)
                if ent:
                    hib = load_results_by_id(ent["results"])['metrics'][method][mk]['higher_is_better']
                    directions[mk] = bool(hib)
                    break
            if directions[mk] is not False:
                break

    # 3) collect rows
    rows: List[Dict[str, Any]] = []
    for m in models:
        for sc in scenarios:
            method = scenarios[sc]
            ent = data.get(method, {}).get(m)
            row = {"model": m.replace("_", r"\_"), "prompt": sc, "metrics": {}, "count": None}
            if ent:
                mets = load_results_by_id(ent["results"])['metrics'][method]
                for mk in metric_keys:
                    row['metrics'][mk] = (mets[mk]['value'], mets[mk]['stderr'])
                row['count'] = mets.get(count_key, {}).get('value', None)
            rows.append(row)

    # 4) best & runner-up
    best_idx: Dict[str, int] = {}
    second_idx: Dict[str, int] = {}
    for mk in metric_keys:
        vals = [(i, r['metrics'][mk][0]) for i, r in enumerate(rows) if mk in r['metrics']]
        # sort with reverse if higher is better
        sorted_vals = sorted(vals, key=lambda x: x[1], reverse=directions[mk])
        if sorted_vals:
            best_idx[mk] = sorted_vals[0][0]
            if len(sorted_vals) > 1:
                second_idx[mk] = sorted_vals[1][0]

    # 5) build LaTeX table
    colspec = 'll' + 'c' * (len(metric_keys) + 1)
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        f"\\begin{{tabular}}{{{colspec}}}",
        r"\toprule",
    ]
    # header
    hdr = ['Model', 'Prompt']
    for mk in metric_keys:
        arrow = '↑' if directions[mk] else '↓'
        hdr.append(f"{rename[mk]} {arrow}")
    hdr.append(rename[count_key])
    lines.append(' & '.join(hdr) + r' \\')
    lines.append(r"\midrule")

    # rows
    for i, r in enumerate(rows):
        cells = [r['model'], r['prompt']]
        for mk in metric_keys:
            if mk in r['metrics']:
                v, se = r['metrics'][mk]
                base = f"{v:.2f}\\,$\\pm$\\,{se:.2f}"
                if best_idx.get(mk) == i:
                    cell = f"\\cellcolor{{green!20}}\\textbf{{{base}}}"
                elif second_idx.get(mk) == i:
                    cell = f"\\cellcolor{{blue!20}}\\textbf{{{base}}}"
                else:
                    cell = base
            else:
                cell = '–'
            cells.append(cell)
        cnt = r['count']
        cells.append(f"{cnt:.1f}" if cnt is not None else '–')
        lines.append(' & '.join(cells) + r' \\')

    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return '\n'.join(lines)


In [11]:


# — example usage:


# define your three scenarios
SCENARIOS = {
    "Ideal": "strokerehab_primitives_1",
    "SP":    "strokerehab_primitives_2",
    "SMC":   "strokerehab_primitives_3",
}

MODELS = [
    "bot_8f", "bot_32f",
    "llava_ov_0p5b", "llava_ov_7b", "llava_ov_72b",
    "llava_next_video_7b", "llava_next_video_72b",
    "internvl_2p5_2b","internvl_2p5_8b","internvl_2p5_38b","internvl_2p5_78b",
    "longva_7b", "nvila_8b", "nvila_15b"
]

METRICS = ["action_error_rate", "edit_score", "mae_avg"]

data = get_latest_run_files("logs")
caption = """\\textcolor{red}{TODO: Run on the 50 video subset.} Results from the video models. The columns represent the action error rate (AER), edit score (ES), and mean absolute error (MAE) for the three prompt methods: Ideal, SP, and SMC. bot\_8f and bot\_32f output "IDLE" for every chunk, or "NO" for each prompt in motion and contact. \\textit{Due to inference costs, the results are reported on a randomly sampled subset of $50$ videos of the original $514$ videos.}"""
latex = generate_model_scenario_table(
    data,
    SCENARIOS,
    MODELS,
    METRICS,
    count_key="count_pred",
    rename={"action_error_rate":"AER","edit_score":"ES","mae_avg":"MAE"},
    caption=caption,
    label="tab:strokerehab_metrics"
)

print(latex)


\begin{table}[ht]
\centering
\small
\caption{\textcolor{red}{TODO: Run on the 50 video subset.} Results from the video models. The columns represent the action error rate (AER), edit score (ES), and mean absolute error (MAE) for the three prompt methods: Ideal, SP, and SMC. bot\_8f and bot\_32f output "IDLE" for every chunk, or "NO" for each prompt in motion and contact. \textit{Due to inference costs, the results are reported on a randomly sampled subset of $50$ videos of the original $514$ videos.}}
\label{tab:strokerehab_metrics}
\begin{tabular}{llcccc}
\toprule
Model & Prompt & AER ↓ & ES ↑ & MAE ↓ & Pred. Count \\
\midrule
bot\_8f & Ideal & 0.92\,$\pm$\,0.03 & 7.69\,$\pm$\,2.56 & 3.53\,$\pm$\,0.80 & 1.0 \\
bot\_8f & SP & 0.92\,$\pm$\,0.03 & 7.69\,$\pm$\,2.56 & 3.53\,$\pm$\,0.80 & 1.0 \\
bot\_8f & SMC & 0.92\,$\pm$\,0.03 & 7.69\,$\pm$\,2.56 & 3.53\,$\pm$\,0.80 & 1.0 \\
bot\_32f & Ideal & 0.92\,$\pm$\,0.03 & 7.69\,$\pm$\,2.56 & 3.53\,$\pm$\,0.80 & 1.0 \\
bot\_32f & SP & 0.92\,$\pm$\

In [ ]:
    "bot_8f", "bot_32f",
    "llava_ov_0p5b", "llava_ov_7b", "llava_ov_72b",
    "llava_next_video_7b", "llava_next_video_72b",
    "internvl_2p5_2b","internvl_2p5_8b","internvl_2p5_38b","internvl_2p5_78b",